In [ ]:
# extract stream from osm

######### collect streams for the four focused cities separately
import osmnx as ox
import geopandas as gpd

place_name = "Dresden, Germany"
# place_name = "Jablonec nad Nisou, Czech Republic"
# place_name = "Poznań, Poland"
# place_name = "Senica, Slovakia"

streams= ox.features_from_place(place_name,tags={"waterway":"stream"})

streams = streams.to_crs(epsg=3857)

print(streams.head())

streams.to_file("dresden_stream.geojson", driver="GeoJSON")

city_boundary = ox.geocode_to_gdf(place_name) 
city_boundary = city_boundary.to_crs(epsg=3857)

# list all the variables in the attribute table
print(streams.columns)

#count the number of the names
print(streams['name'].value_counts())

# dissolve stream geometries by name
streams_dissolved = streams.dissolve(by='name')

streams_dissolved = streams_dissolved.reset_index()
streams_dissolved.to_file("dresden_stream_dissolved.geojson", driver="GeoJSON")

                                                          geometry waterway  \
element id                                                                    
way     4398867  LINESTRING (1513425.124 6628739.185, 1513434.6...   stream   
        4402184  LINESTRING (1519194.334 6627859.936, 1519259.1...   stream   
        4417709  LINESTRING (1522992.244 6628922.84, 1523005.80...   stream   
        4863060  LINESTRING (1538245.051 6620989.339, 1538293.1...   stream   
        4940315  LINESTRING (1540491.245 6633242.246, 1540419.0...   stream   

                              name wikidata  \
element id                                    
way     4398867  Zschonergrundbach      NaN   
        4402184        Gorbitzbach      NaN   
        4417709        Gorbitzbach      NaN   
        4863060       Lockwitzbach      NaN   
        4940315    Mordgrundwasser      NaN   

                                                              note boat  \
element id                               

In [34]:
# plot Geberbach at 4 Scales

import matplotlib.pyplot as plt
import contextily as ctx

# filter Geberbach
geberbach = streams_dissolved[streams_dissolved['name'].str.contains("Geberbach", case=False, na=False)]

# count the number of the names
print(geberbach['name'].value_counts())
# catchment (1000m buffer and envelope)
catchment_buffer = geberbach.buffer(1000).geometry.envelope # rectangle around the stream buffer
catchment_buffer1 = geberbach.buffer(1000)
catchment_gdf = gpd.GeoDataFrame(geometry=catchment_buffer, crs=geberbach.crs)
catchment1_gdf = gpd.GeoDataFrame(geometry=catchment_buffer1, crs=geberbach.crs)

# stream Corridor (50m buffer)
corridor_buffer = geberbach.buffer(50)
corridor_gdf = gpd.GeoDataFrame(geometry=corridor_buffer, crs=geberbach.crs)

# pilot Channel (5m buffer)
pilot_buffer = geberbach.buffer(5)
pilot_gdf = gpd.GeoDataFrame(geometry=pilot_buffer, crs=geberbach.crs)


name
Geberbach    1
Name: count, dtype: int64


In [32]:
geometrien = gpd.read_file("7 Geometrien.shp")

In [ ]:
import folium
import geopandas as gpd
from shapely.geometry import mapping

# convert to WGS84 for folium
streams_dissolved_wgs = streams_dissolved.to_crs(epsg=4326)
geberbach_wgs = geberbach.to_crs(epsg=4326)
catchment_wgs = catchment_gdf.to_crs(epsg=4326)
catchment1_wgs = catchment1_gdf.to_crs(epsg=4326)
corridor_wgs = corridor_gdf.to_crs(epsg=4326)
pilot_wgs = pilot_gdf.to_crs(epsg=4326)
city_boundary_wgs = city_boundary.to_crs(epsg=4326)
geometrien_wgs = geometrien.to_crs(epsg=4326)

# get centroid to center the map
center = geberbach_wgs.geometry.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=14, tiles="CartoDB Positron")

# function
def add_gdf_to_map(gdf, name, color, alpha=0.3, weight=2):
    folium.GeoJson(
        data=gdf.__geo_interface__,
        name=name,
        style_function=lambda x: {
            "color": color,
            "weight": weight,
            "fillColor": color,
            "fillOpacity": alpha
        }
    ).add_to(m)

# add each layer
add_gdf_to_map(streams_dissolved_wgs, "All Streams", "lightblue", alpha=0.6, weight=3)
add_gdf_to_map(catchment_wgs, "Catchment (rectangle)", "grey", alpha=0.2)
add_gdf_to_map(catchment1_wgs, "Catchment (~1000m)", "grey", alpha=0.1)
add_gdf_to_map(corridor_wgs, "Stream Corridor (~50m)", "darkpink", alpha=0.4)
add_gdf_to_map(pilot_wgs, "Smaller buffer (~5m)", "blue", alpha=0.4)
add_gdf_to_map(geberbach_wgs, "Geberbach Stream", "darkblue", alpha=1.0, weight=3)
add_gdf_to_map(city_boundary_wgs, "City Boundary", "black", alpha=0.0, weight=2)
add_gdf_to_map(geometrien_wgs, "7 Geometrien(Pilot Channel))", "orange", alpha=0.6, weight=2)


# add layer control
folium.LayerControl().add_to(m)

# save to HTML
m.save("geberbach_map.html")
print("Map saved")

Map saved


/var/folders/58/86zdkmzd6y3d82s16dcq774cn10vxp/T/ipykernel_34544/3102894920.py:16: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = geberbach_wgs.geometry.unary_union.centroid
